# 🏥 ViBioMIR - Google Colab High-Speed Retrieval & Submission Pipeline

Notebook này được tối ưu hóa để chạy trực tiếp trên **Google Colab (Tesla T4 / L4 / A100)** nhằm:
1. **Tận dụng VRAM 16GB - 40GB**: Chạy batch size lớn (64–128) với `BAAI/bge-m3` và `BAAI/bge-reranker-large` (nhanh hơn 3x–8x so với GPU laptop 6GB).
2. **Tận dụng băng thông Datacenter (1Gbps+)**: Đồng bộ các shard Parquet từ Hugging Face Storage Bucket `hf://buckets/nieths/ViBioMIR/corpus` chỉ trong vài chục giây.
3. **Tối ưu hóa tham số cho độ đo Macro F2** (`top_k_docs: 25`, `top_k_chunks: 40`, `hybrid_top_k: 100`).
4. **Tự động đóng gói & tải xuống** file `submission.zip` chuẩn để nộp trực tiếp lên Leaderboard.

## 1. Kiểm tra GPU và Cài đặt Môi trường

In [ ]:
!nvidia-smi

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate faiss-gpu pyarrow "huggingface_hub[cli]" rank-bm25 pyvi loguru typer pydantic pyyaml tqdm

## 2. Clone hoặc Cập nhật Codebase từ GitHub

In [ ]:
import os
from pathlib import Path

# Nhập link repo GitHub của bạn (nếu là private repo, dùng token: https://<token>@github.com/<user>/<repo>.git)
REPO_URL = "https://github.com/nieths/med-doc-retrieval.git"  # Thay bằng repo của bạn nếu cần
PROJECT_DIR = Path("/content/med-doc-retrieval")

if not PROJECT_DIR.exists():
    !git clone $REPO_URL
else:
    %cd $PROJECT_DIR
    !git pull

%cd $PROJECT_DIR
print("Current directory:", os.getcwd())

## 3. Đồng bộ Dữ liệu từ Hugging Face Storage Bucket

Đồng bộ toàn bộ các shard Parquet đã crawl từ bucket `hf://buckets/nieths/ViBioMIR/corpus` về máy Colab:

In [ ]:
# Nhập Hugging Face Token nếu bucket là private (có thể bỏ trống nếu bucket public)
import os
from huggingface_hub import login

HF_TOKEN = ""  # Điền hf_... nếu bucket private
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN

!mkdir -p data/processed/parquet_corpus
!hf sync hf://buckets/nieths/ViBioMIR/corpus data/processed/parquet_corpus

!ls -lh data/processed/parquet_corpus

## 4. Chạy Pipeline: Xây dựng Index FAISS/BM25 & Inference (Batch Size 64)

Lệnh sau sẽ:
- Chunking tài liệu với độ dài tối ưu 800 chars (overlap 128 chars).
- Tính toán Embedding BGE-M3 trên GPU với batch size 64.
- Hybrid Search (Dense + BM25) với `hybrid_top_k: 100`.
- Reranking với `BAAI/bge-reranker-large`.
- Trích xuất `top_k_docs: 25` và `top_k_chunks: 40` để tối đa hóa điểm Macro F2.

In [ ]:
!PYTHONPATH=. python scripts/colab_runner.py \
    --corpus-dir data/processed/parquet_corpus \
    --queries data/raw/queries.jsonl \
    --output-name submission_v2_colab.json \
    --batch-size 64 \
    --skip-sync

## 5. Tải file `submission.zip` về máy để nộp Leaderboard

In [ ]:
from google.colab import files
from pathlib import Path

zip_path = Path("outputs/submissions/submission_v2_colab.zip")
if zip_path.exists():
    print(f"Downloading {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)...")
    files.download(str(zip_path))
else:
    print("File submission chưa tồn tại. Vui lòng kiểm tra lại bước chạy pipeline!")